In [25]:
import cv2
import numpy as np
import os


INPUT_DIR = "input"
OUTPUT_DIR = "output"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)



def get_edges(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    
    med_val = np.median(blurred)
    low_th = int(max(0, 0.7 * med_val))
    high_th = int(min(255, 1.4 * med_val))

    edge_img = cv2.Canny(blurred, low_th, high_th)

    
    kernel = np.ones((5, 5), np.uint8)
    edge_img = cv2.dilate(edge_img, kernel, iterations=1)
    edge_img = cv2.morphologyEx(edge_img, cv2.MORPH_CLOSE, kernel, iterations=2)

    return edge_img



def analyze_contour(cnt):
    area = cv2.contourArea(cnt)

    if area < 800 or area > 200000:
        return None

    perimeter = cv2.arcLength(cnt, True)
    if perimeter == 0:
        return None

    circularity = 4 * np.pi * area / (perimeter ** 2)

    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)

    if hull_area == 0:
        return None

    solidity = area / hull_area

    return circularity, solidity



def get_label(circularity, solidity):
    if circularity > 0.75 and solidity > 0.50:
        return "Intact Biscuit", (0, 255, 0)
    else:
        return "Broken Biscuit", (0, 0, 255)



def process_images():
    for file_name in os.listdir(INPUT_DIR):

        if not file_name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        img_path = os.path.join(INPUT_DIR, file_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.resize(img, (800, 800))

        edges = get_edges(img)

        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        intact_count = 0
        broken_count = 0

        for cnt in contours:

            x, y, w, h = cv2.boundingRect(cnt)

            if w > 750 or h > 750:
                continue

            result = analyze_contour(cnt)

            if result is None:
                continue

            circ, solid = result
            label, color = get_label(circ, solid)

            if "Intact" in label:
                intact_count += 1
            else:
                broken_count += 1

            
            cv2.drawContours(img, [cnt], -1, color, 2)
            cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)

            cv2.putText(img, label, (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        
        summary_text = f"Intact: {intact_count}  Broken: {broken_count}"
        cv2.putText(img, summary_text, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

        
        save_path = os.path.join(OUTPUT_DIR, "result" + file_name)
        cv2.imwrite(save_path, img)

    print("Processing finished. Check 'output' folder.")


if __name__ == "__main__":
    process_images()

Processing finished. Check 'output' folder.
